# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides users through loading, exploring, and performing analysis on the FAIR² dataset "Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution," using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

## Dataset Source
This dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

**https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json**

All fields and datasets are referenced by their `@id` identifiers for maximum reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View top-level metadata fields
meta = dataset.metadata  # metadata is an object, not a dictionary
print(f"Dataset loaded: {meta.name}\n\n{meta.description}")

## 2. Data Overview

Let's explore the structure of the dataset, including available record sets and their associated fields and columns. All references are made by their `@id`.

In [ ]:
# List available record sets
record_sets = dataset.metadata.record_sets  # This is a list of records sets metadata objects
print(f"Number of record sets: {len(record_sets)}\n")

for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id      : {rs['@id']}")
    print(f"  name     : {rs.get('name', '(no name)')}")
    print(f"  desc.    : {rs.get('description', '(no description)')}")
    # List fields for each record set
    if 'fields' in rs:
        print(f"  Fields:")
        for f in rs['fields']:
            print(f"    - {f['@id']}: {f.get('name', '')} [{f.get('dataType', '')}]")
    print()

## 3. Data Extraction

We'll load data from one or more record sets into pandas DataFrames for exploratory analysis. Please refer to the `@id` of the record sets you want to analyze (as found above).

In [ ]:
# Fill in with discovered record set @ids (here assuming only one, based on metadata)

# Example: If the main record set's @id is 'cr:RecordSet:second_primary_crc', replace accordingly
# We'll dynamically detect them from dataset.metadata.record_sets

all_record_set_ids = [recset['@id'] for recset in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame shape: {dataframes[record_set_id].shape}")

# Display the list of columns for the first main record set as an example
if all_record_set_ids:
    main_record_set_id = all_record_set_ids[0]
    print(f"\nColumns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll select numeric and categorical fields by their `@id` for filtering, normalization, and grouping. 

Make sure to reference fields using their `@id`, as in the overview above (e.g., `'cr:Field:age'`).

**Example operations:** Filtering on a numeric variable—for instance, `age` > 60.

In [ ]:
# Example variable assignments; update with available field @ids from previous cells if necessary
numeric_field = None
group_field = None
main_df = dataframes.get(main_record_set_id)

# Try to automatically find a field named 'age' (case-insensitive)
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field = col
        break

# Similarly, try finding a good categorical grouping field (e.g., 'sex', 'msi', 'location')
for col in main_df.columns:
    if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower():
        group_field = col
        break

if numeric_field is not None:
    print(f"Numeric field selected: {numeric_field}")
    
    # Set a threshold for filtering (e.g., age > 60)
    threshold = 60
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in main_df.columns:
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        display(grouped_df)
else:
    print("No numeric field (e.g., 'age') found in the main record set for EDA.")

## 5. Visualization

Let's visualize the distribution of the numeric field and, if grouped, show group-wise statistics.

We'll use built-in plotting, assuming `age` was found as an example.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field], bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and analyze a FAIR² Croissant dataset using the `mlcroissant` library.

- All dataset and field references were made using their `@id` identifiers.
- We visualized the distribution and grouped means of the key numeric field (e.g., age), and showed how to filter and normalize the data.

You can further extend this notebook by exploring more record sets, fields, or running your own custom analysis using the field `@id`s!